In [71]:
import ROOT as r
import math

In [ ]:
class MyMuon(r.TLorentzVector):
    def __init__(self, px=0, py=0, pz=0, e=0):
        super().__init__(px, py, pz, e)
        self.isolation = 0.0
        self.charge = 0

    def SetIsolation(self, x):
        self.isolation = x

    def SetCharge(self, q):
        self.charge = q

    def GetIsolation(self):
        return self.isolation

    def IsIsolated(self, relcut=0.05):
        if self.Pt() == 0:
            return False
        return (self.isolation / self.Pt()) < relcut

    def GetCharge(self):
        return self.charge


In [ ]:
class Plotter:
    def __init__(self):
        """Constructor equivalent to Plotter::Plotter()."""
        self.data = []         
        self.bg = []            
        self.signal = []        
        self.data_names = []
        self.bg_names = []
        self.signal_names = []
        self.N_histos = 0

    def __del__(self):
        """Destructor (handled automatically in Python)."""
        pass

    def SetData(self, v, name):
        """
        Add a set of data histograms.
        v: list of TH1F objects
        name: string label for this dataset
        """
        self.data.append(v)
        self.data_names.append(name)
        self.N_histos = len(v)

    def ClearData(self):
        self.data.clear()
        self.data_names.clear()

    def AddBg(self, v, name):
        """Add a set of background histograms."""
        self.bg.append(v)
        self.bg_names.append(name)
        self.N_histos = len(v)

    def ClearBg(self):
        self.bg.clear()
        self.bg_names.clear()

    def AddSig(self, v, name):
        """Add a set of signal histograms."""
        self.signal.append(v)
        self.signal_names.append(name)
        self.N_histos = len(v)

    def ClearSig(self):
        self.signal.clear()
        self.signal_names.clear()

    def Plot(self, filename="result.pdf"):
        """
        Reproduces the C++ Plotter::Plot() method.
        Makes one canvas per histogram index, stacking BG, plotting signal + data.
        """
        if self.N_histos == 0:
            print("No histograms to plot.")
            return

        r.gStyle.SetOptStat(0)
        r.gROOT.SetBatch(True)

        canvases = []
        for i in range(self.N_histos):
            c = r.TCanvas(f"c{i}", f"Canvas {i}", 800, 600)
            canvases.append(c)

            stack = r.THStack(f"stack_{i}", f"Histogram {i}")
            legend = r.TLegend(0.7, 0.7, 0.9, 0.9)
            legend.SetBorderSize(0)
            legend.SetFillStyle(0)

            for j, bg_set in enumerate(self.bg):
                h = bg_set[i]
                h.SetFillColor(r.kAzure + j)
                h.SetLineColor(r.kBlack)
                stack.Add(h)
                legend.AddEntry(h, self.bg_names[j], "f")

            stack.Draw("hist")
            stack.GetXaxis().SetTitle(self.data_names[0] if self.data_names else "")
            stack.GetYaxis().SetTitle("Events")

            # --- Overlay signals ---
            for j, sig_set in enumerate(self.signal):
                hsig = sig_set[i]
                hsig.SetLineColor(r.kRed + j)
                hsig.SetLineWidth(2)
                hsig.Draw("hist same")
                legend.AddEntry(hsig, self.signal_names[j], "l")

            for j, data_set in enumerate(self.data):
                hdata = data_set[i]
                hdata.SetMarkerStyle(20)
                hdata.SetMarkerColor(r.kBlack)
                hdata.Draw("E same")
                legend.AddEntry(hdata, self.data_names[j], "lep")

            legend.Draw()
            c.Update()

        canvases[0].Print(f"{filename}(")
        for c in canvases[1:-1]:
            c.Print(filename)
        canvases[-1].Print(f"{filename})")

        print(f"Saved plots to {filename}")


'\nclass Plotter:\n    def __init__(self):\n        """Constructor equivalent to Plotter::Plotter()."""\n        self.data = []          # list of list of TH1F\n        self.bg = []            # background histograms\n        self.signal = []        # signal histograms\n        self.data_names = []\n        self.bg_names = []\n        self.signal_names = []\n        self.N_histos = 0\n\n    def __del__(self):\n        """Destructor (handled automatically in Python)."""\n        pass\n\n    # ---- Data handling ----\n    def SetData(self, v, name):\n        """\n        Add a set of data histograms.\n        v: list of TH1F objects\n        name: string label for this dataset\n        """\n        self.data.append(v)\n        self.data_names.append(name)\n        self.N_histos = len(v)\n\n    def ClearData(self):\n        self.data.clear()\n        self.data_names.clear()\n\n    # ---- Background handling ----\n    def AddBg(self, v, name):\n        """Add a set of background histograms

In [ ]:
class Plotter:
    def __init__(self):
        self.data = []          # list of list of TH1F
        self.bg = []            # list of list of TH1F (background)
        self.signal = []        # list of list of TH1F (signal)
        self.data_names = []
        self.bg_names = []
        self.signal_names = []
        self.N_histos = 0

    def __del__(self):
        pass

    def SetData(self, v, name):
        self.data.append(v)
        self.data_names.append(name)
        self.N_histos = len(v)

    def ClearData(self):
        self.data.clear()
        self.data_names.clear()

    def AddBg(self, v, name):
        self.bg.append(v)
        self.bg_names.append(name)
        self.N_histos = len(v)

    def ClearBg(self):
        self.bg.clear()
        self.bg_names.clear()

    def AddSig(self, v, name):
        self.signal.append(v)
        self.signal_names.append(name)
        self.N_histos = len(v)

    def ClearSig(self):
        self.signal.clear()
        self.signal_names.clear()

    def Plot(self, filename="result.pdf"):
        r.gROOT.Reset()

        MyStyle = r.TStyle("MyStyle", "My Root Styles")
        MyStyle.SetStatColor(0)
        MyStyle.SetCanvasColor(0)
        MyStyle.SetPadColor(0)
        MyStyle.SetPadBorderMode(0)
        MyStyle.SetCanvasBorderMode(0)
        MyStyle.SetFrameBorderMode(0)
        MyStyle.SetOptStat(0)
        MyStyle.SetStatBorderSize(2)
        MyStyle.SetOptTitle(0)
        MyStyle.SetPadTickX(1)
        MyStyle.SetPadTickY(1)
        MyStyle.SetPadBorderSize(2)
        MyStyle.SetPalette(51, 0)
        MyStyle.SetPadBottomMargin(0.15)
        MyStyle.SetPadTopMargin(0.05)
        MyStyle.SetPadLeftMargin(0.15)
        MyStyle.SetPadRightMargin(0.25)
        MyStyle.SetTitleColor(1)
        MyStyle.SetTitleFillColor(0)
        MyStyle.SetTitleFontSize(0.05)
        MyStyle.SetTitleBorderSize(0)
        MyStyle.SetLineWidth(1)
        MyStyle.SetHistLineWidth(3)
        MyStyle.SetLegendBorderSize(0)
        MyStyle.SetNdivisions(502, "x")
        MyStyle.SetMarkerSize(0.8)
        MyStyle.SetTickLength(0.03)
        MyStyle.SetTitleOffset(1.5, "x")
        MyStyle.SetTitleOffset(1.5, "y")
        MyStyle.SetTitleOffset(1.0, "z")
        MyStyle.SetLabelSize(0.05, "x")
        MyStyle.SetLabelSize(0.05, "y")
        MyStyle.SetLabelSize(0.05, "z")
        MyStyle.SetLabelOffset(0.03, "x")
        MyStyle.SetLabelOffset(0.03, "y")
        MyStyle.SetLabelOffset(0.03, "z")
        MyStyle.SetTitleSize(0.05, "x")
        MyStyle.SetTitleSize(0.05, "y")
        MyStyle.SetTitleSize(0.05, "z")

        r.gROOT.SetStyle("MyStyle")

        DrawLog = True

        for i in range(self.N_histos):
            hs = None
            Nset = len(self.data) + len(self.bg) + len(self.signal)
            if Nset > 20:
                Nset = 20

            l = r.TLegend(0.76, 0.95 - 0.8 * Nset / 20.0, 1.0, 0.95)
            l.SetFillStyle(1001)
            l.SetFillColor(r.kWhite)
            l.SetLineColor(r.kWhite)
            l.SetLineWidth(2)

            if len(self.bg) > 0:
                hs = r.THStack("hs", self.bg[0][i].GetName())
                for j, bg_set in enumerate(self.bg):
                    h = bg_set[i]
                    color_map = [
                        r.kRed, r.kOrange, r.kYellow, r.kGreen, r.kCyan,
                        r.kBlue, r.kMagenta, r.kGray, r.kGray + 2
                    ]
                    h.SetFillColor(color_map[j] if j < len(color_map) else r.kBlack)
                    hs.Add(h)
                    l.AddEntry(h, self.bg_names[j], "f")

            c = r.TCanvas(f"c{i}", f"Canvas {i}", 800, 600)
            c.SetLogy(DrawLog)

            plotname = ""

            if len(self.data) > 0:
                plotname = self.data[0][i].GetName()
                hdata = self.data[0][i]
                hdata.SetMaximum(5 * hdata.GetMaximum())
                hdata.GetXaxis().SetTitleOffset(1.3)
                hdata.GetYaxis().SetTitleOffset(1.3)
                hdata.GetYaxis().SetTitle("Events")
                hdata.GetXaxis().SetNdivisions(505)
                hdata.Draw("")
                l.AddEntry(hdata, self.data_names[0], "p")

                if len(self.bg) > 0:
                    hs.Draw("histsame")

                hdata.SetMarkerStyle(20)
                hdata.Draw("psame")
                l.Draw("same")

            elif len(self.data) == 0 and len(self.bg) > 0:
                plotname = self.bg[0][i].GetName()
                hs.Draw("hist")
                hs.GetXaxis().SetTitleOffset(1.3)
                hs.GetXaxis().SetNdivisions(505)
                hs.GetYaxis().SetTitleOffset(1.3)
                hs.GetYaxis().SetTitle("Events")
                hs.GetXaxis().SetTitle(self.bg[0][i].GetXaxis().GetTitle())
                l.Draw("same")

            if i == 0 and self.N_histos > 1:
                c.Print(f"{filename}(")
            elif i > 0 and i == self.N_histos - 1:
                c.Print(f"{filename})")
            else:
                c.Print(filename)

        print(f"Plots saved to {filename}")


In [ ]:
class MyAnalysis:
    def __init__(self, tree):
        self.tree = tree
        self.Muons = []

        # vectors
        self.hadB = r.TLorentzVector()
        self.lepB = r.TLorentzVector()
        self.hadWq = r.TLorentzVector()
        self.hadWqb = r.TLorentzVector()
        self.lepWl = r.TLorentzVector()
        self.lepWn = r.TLorentzVector()
        self.met = r.TLorentzVector()

        self.histograms = []
        self.histograms_MC = []
        self.EventWeight = 1.0
        self.weight_factor = 1.0

    def BuildEvent(self):
        """Equivalent of C++ MyAnalysis::BuildEvent()."""
        self.Muons.clear()

        NMuon = int(self.tree.NMuon)
        for i in range(NMuon):
            muon = MyMuon(
                self.tree.Muon_Px[i],
                self.tree.Muon_Py[i],
                self.tree.Muon_Pz[i],
                self.tree.Muon_E[i]
            )
            muon.SetIsolation(self.tree.Muon_Iso[i])
            muon.SetCharge(self.tree.Muon_Charge[i])
            self.Muons.append(muon)


    def BuildMCParticles(self):
        t = self.tree

        self.hadB.SetXYZM(t.MChadronicBottom_px,  t.MChadronicBottom_py,  t.MChadronicBottom_pz,  4.8)
        self.lepB.SetXYZM(t.MCleptonicBottom_px,  t.MCleptonicBottom_py,  t.MCleptonicBottom_pz,  4.8)
        self.hadWq.SetXYZM(t.MChadronicWDecayQuark_px,     t.MChadronicWDecayQuark_py,     t.MChadronicWDecayQuark_pz,     0.0)
        self.hadWqb.SetXYZM(t.MChadronicWDecayQuarkBar_px, t.MChadronicWDecayQuarkBar_py, t.MChadronicWDecayQuarkBar_pz,  0.0)
        self.lepWl.SetXYZM(t.MClepton_px,  t.MClepton_py,  t.MClepton_pz,  0.0)
        self.lepWn.SetXYZM(t.MCneutrino_px, t.MCneutrino_py, t.MCneutrino_pz, 0.0)
        self.met.SetXYZM(t.MET_px, t.MET_py, 0.0, 0.0)

        self.EventWeight *= self.weight_factor

    def Begin(self):
        """Called at the start of processing."""
        option = getattr(self, "option", "")
        # Nothing else needed; can add setup logic here

    def SlaveBegin(self):
        """Called after Begin(), to define histograms."""
        option = getattr(self, "option", "")

        # histograms
        self.h_Mmumu = r.TH1F("Mmumu", "Invariant di-muon mass", 60, 60, 120)
        self.h_Mmumu.GetXaxis().SetTitle("m_{#mu#mu}")
        self.h_Mmumu.Sumw2()

        self.histograms.append(self.h_Mmumu)
        self.histograms_MC.append(self.h_Mmumu)

class MyAnalysis:
    def __init__(self, tree):
        self.tree = tree

        self.hadB = r.TLorentzVector()
        self.lepB = r.TLorentzVector()
        self.hadWq = r.TLorentzVector()
        self.hadWqb = r.TLorentzVector()
        self.lepWl = r.TLorentzVector()
        self.lepWn = r.TLorentzVector()
        self.met = r.TLorentzVector()

        self.histograms = []
        self.histograms_MC = []
        self.EventWeight = 1.0
        self.weight_factor = 1.0 

    def BuildMCParticles(self):
        t = self.tree

        self.hadB.SetXYZM(t.MChadronicBottom_px,  t.MChadronicBottom_py,  t.MChadronicBottom_pz,  4.8)
        self.lepB.SetXYZM(t.MCleptonicBottom_px,  t.MCleptonicBottom_py,  t.MCleptonicBottom_pz,  4.8)
        self.hadWq.SetXYZM(t.MChadronicWDecayQuark_px,     t.MChadronicWDecayQuark_py,     t.MChadronicWDecayQuark_pz,     0.0)
        self.hadWqb.SetXYZM(t.MChadronicWDecayQuarkBar_px, t.MChadronicWDecayQuarkBar_py, t.MChadronicWDecayQuarkBar_pz,  0.0)
        self.lepWl.SetXYZM(t.MClepton_px,  t.MClepton_py,  t.MClepton_pz,  0.0)
        self.lepWn.SetXYZM(t.MCneutrino_px, t.MCneutrino_py, t.MCneutrino_pz, 0.0)
        self.met.SetXYZM(t.MET_px, t.MET_py, 0.0, 0.0)

        self.EventWeight *= self.weight_factor

    def Begin(self):
        """Called at the start of processing."""
        option = getattr(self, "option", "")
        
    def SlaveBegin(self):
        """Called after Begin(), to define histograms."""
        option = getattr(self, "option", "")

        self.h_Mmumu = r.TH1F("Mmumu", "Invariant di-muon mass", 60, 60, 120)
        self.h_Mmumu.GetXaxis().SetTitle("m_{#mu#mu}")
        self.h_Mmumu.Sumw2()

        self.histograms.append(self.h_Mmumu)
        self.histograms_MC.append(self.h_Mmumu)
    def SlaveBegin(self):
        """Called after Begin(), to define histograms."""
        option = getattr(self, "option", "")

        self.h_Mmumu = r.TH1F("Mmumu", "Invariant di-muon mass", 60, 60, 120)
        self.h_Mmumu.GetXaxis().SetTitle("m_{#mu#mu}")
        self.h_Mmumu.Sumw2()

        self.h_NMuon = r.TH1F("NMuon", "Number of muons", 7, 0, 7)
        self.h_NMuon.GetXaxis().SetTitle("No. Muons")
        self.h_NMuon.Sumw2()

        self.histograms.extend([self.h_Mmumu, self.h_NMuon])
        self.histograms_MC.extend([self.h_Mmumu, self.h_NMuon])
        

In [ ]:
# Exercise 1: Invariant Di-Muon mass

EventWeight = 1.0
triggerIsoMu24 = True
MuonPtCut = 25.0
MuonRelIsoCut = 0.10

N_IsoMuon = 0       
muon1 = None        
muon2 = None        

h_NMuon = r.TH1F("NMuon", "Number of muons", 7, 0, 7)
h_Mmumu = r.TH1F("Mmumu", "Invariant di-muon mass", 60, 60, 120)

N_IsoMuon = 0
muon1 = None
muon2 = None

for muon in analysis.Muons:
    if muon.IsIsolated(MuonRelIsoCut):
        N_IsoMuon += 1
        if N_IsoMuon == 1:
            muon1 = muon
        elif N_IsoMuon == 2:
            muon2 = muon

h_NMuon.Fill(N_IsoMuon, EventWeight)

if N_IsoMuon > 1 and triggerIsoMu24:
    if muon1.Pt() > MuonPtCut:
        h_Mmumu.Fill((muon1 + muon2).M(), EventWeight)

c = r.TCanvas()
h_Mmumu.Draw()
c.Draw()

NameError: name 'analysis' is not defined

Warning in <TROOT::Append>: Replacing existing TH1: NMuon (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: Mmumu (Potential memory leak).
